DHANSHREE | 221A033 | 08 | BE-AIDS

In [1]:
import numpy as np
import random
#GridWorld Environment.
class Gridworld:
    def __init__(self, size=4, start=(0, 0), goal=(3, 3)): # Fixed: __init__ and goal parameter
        self.size = size
        self.start = start
        self.goal = goal # Fixed: assignment
        self.state = start
        self.actions = [(0, 1), (1, 0), (0, -1), (-1, 0)] # Right, Down, Left, Up (Fixed: - 1 to -1, 8 to 0)

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        # Calculate next state based on current state and action
        next_state = (self.state[0] + self.actions[action][0],
                      self.state[1] + self.actions[action][1])

        # Check for boundary conditions
        next_state_row, next_state_col = next_state
        if 0 <= next_state_row < self.size and 0 <= next_state_col < self.size:
            self.state = next_state
        # If next_state is out of bounds, the agent stays in the current state (can be modified for penalty)

        # Determine reward and if the episode is done
        reward = -0.1 # Default reward for each step
        done = False
        if self.state == self.goal:
            reward = 1 # Reward for reaching goal
            done = True
        return self.state, reward, done

\**2. Monte Carlo Control with Exploring Starts

This is a model-free reinforcement learning algorithm that learns by sampling episodes and averaging rewards.** **bold text**

In [2]:
def monte_carlo_control(env, episodes=5000, gamma=0.9, epsilon=0.1):
    # Initialize Q-table with zeros for all state-action pairs
    Q = {((r, c), a): 0.0 for r in range(env.size) for c in range(env.size) for a in range(4)}
    # Initialize returns (list of returns for each state-action pair) for averaging
    returns = {((r, c), a): [] for r in range(env.size) for c in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        episode = [] # List to store (state, action, reward) tuples for the current episode

        # Generate an episode using an epsilon-greedy policy with exploring starts
        while True:
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = random.choice(range(4)) # Explore: choose a random action
            else:
                # Exploit: choose action with the highest Q-value for the current state
                action = max(range(4), key=lambda a: Q[(state, a)])

            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state

            if done:
                break

        # Update Q-values using the episode's returns (First-Visit Monte Carlo)
        G = 0 # Initialize return for the current episode
        visited_sa = set() # To keep track of state-action pairs already visited in this episode

        # Iterate through the episode in reverse to calculate returns
        for t in reversed(range(len(episode))):
            state_t, action_t, reward_t = episode[t]
            G = reward_t + gamma * G # Correct calculation for discounted returns

            # Only update Q for the first time this (state, action) pair is visited in the episode
            if (state_t, action_t) not in visited_sa:
                returns[(state_t, action_t)].append(G)
                Q[(state_t, action_t)] = np.mean(returns[(state_t, action_t)])
                visited_sa.add((state_t, action_t)) # Mark as visited

    return Q

**3. SARSA (State-Action-Reward-State-Action)**

This is a Temporal Difference (TD) learning algorithm.

In [3]:
#Temporal Difference Learning (SARSA)
def sarsa(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    # Initialize Q-table with zeros for all state-action pairs
    Q = {((r, c), a): 0.0 for r in range(env.size) for c in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        # Choose action A from state S using an epsilon-greedy policy
        if random.random() < epsilon:
            action = random.choice(range(4)) # Explore
        else:
            action = max(range(4), key=lambda a: Q[(state, a)]) # Exploit

        while True:
            next_state, reward, done = env.step(action)

            # Choose next action A' from next state S' using an epsilon-greedy policy
            if random.random() < epsilon:
                next_action = random.choice(range(4)) # Explore
            else:
                next_action = max(range(4), key=lambda a: Q[(next_state, a)]) # Exploit

            # SARSA Update Rule
            Q[(state, action)] = Q[(state, action)] + alpha * (reward + gamma * Q[(next_state, next_action)] - Q[(state, action)])

            state = next_state
            action = next_action

            if done:
                break
    return Q

**4. Q-Learning is very similar to SARSA but uses max future Q-value instead of following the policy.**

In [4]:
#Q-Learning Algorithm
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.9, epsilon=0.1):
    # Initialize Q-table with zeros for all state-action pairs
    Q = {((r, c), a): 0.0 for r in range(env.size) for c in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        while True:
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = random.choice(range(4)) # Explore
            else:
                # Exploit: choose action with the highest Q-value for the current state
                action = max(range(4), key=lambda a: Q[(state, a)])

            next_state, reward, done = env.step(action)

            # Q-Learning Update Rule (off-policy: uses max future Q-value)
            max_future_q = max(Q[(next_state, a)] for a in range(4))
            Q[(state, action)] += alpha * (reward + gamma * max_future_q - Q[(state, action)])

            state = next_state
            if done:
                break
    return Q

**5. Running the Training
Monte Carlo Control, SARSA, and Q-Learning are trained on the GridWorld environment.**

In [5]:
# Initialize Environment
env = Gridworld()
#Train with Monte Carlo
Q_mc = monte_carlo_control(env)
#Train with SARSA
Q_sarsa = sarsa(env)
# Train with Q-Learning
Q_qlearning = q_learning(env)

**6. Sample Q-values**


At the end, the script prints sample Q-values to compare the different learning methods:

In [6]:
#Print Sample Q-values
print("Sample Q-values from Monte Carlo:", {k: Q_mc[k] for k in list(Q_mc.keys())[:20]})
print("Sample Q-values from SARSA:", {k: Q_sarsa[k] for k in list(Q_sarsa.keys())[:20]})
print("Sample Q-values from Q-Learning:", {k: Q_qlearning[k] for k in list(Q_qlearning.keys())[:20]})

Sample Q-values from Monte Carlo: {((0, 0), 0): np.float64(0.09992038326439757), ((0, 0), 1): np.float64(0.12745191630498423), ((0, 0), 2): np.float64(0.0134047444850001), ((0, 0), 3): np.float64(0.010368517268970687), ((0, 1), 0): np.float64(-0.19877678438276877), ((0, 1), 1): np.float64(0.2564270668918921), ((0, 1), 2): np.float64(-0.006205329999999902), ((0, 1), 3): np.float64(0.1809800000000001), ((0, 2), 0): np.float64(-0.4660939652950418), ((0, 2), 1): np.float64(0.4366907692307694), ((0, 2), 2): 0.0, ((0, 2), 3): np.float64(0.31220000000000014), ((0, 3), 0): np.float64(-0.993913494555659), ((0, 3), 1): np.float64(-0.9932372161729545), ((0, 3), 2): np.float64(0.24659000000000014), ((0, 3), 3): 0.0, ((1, 0), 0): np.float64(0.2550509810854239), ((1, 0), 1): np.float64(0.2635490548908085), ((1, 0), 2): np.float64(0.1265368220000001), ((1, 0), 3): np.float64(0.003419926193970687)}
Sample Q-values from SARSA: {((0, 0), 0): 0.08886994841975615, ((0, 0), 1): 0.10473616150693021, ((0, 0)